# 6. Forecasting episode duration and peak SST (RQ2 and RQ3)

**What it does**
- **Duration (Section 4):** once an episode is confirmed (its 3rd month, plus 2 months for ICEN to be published), forecasts how many months it has left. Compares the historical median, linear regression on ICEN at confirmation, ARIMA(1,0,1), ETS and BSTS.
- **Peak SST (Section 5):** uses ARIMA's predicted end as the horizon, fits a GAM per lead time (recent SST anomalies, season and a curved effect of ICEN) and takes the highest (El Niño) or lowest (La Niña) point of the path. Compares it with persistence and the historical median.
- Every method uses only information available on the forecast date and is scored on the same episodes (15 for duration, 5 for peak), with MAE and RMSE.
- Both sections are in one notebook because the peak forecast reuses the duration code and its ARIMA forecasts.

**Inputs**
- `data/processed/icen_monthly_pm04.csv` from notebook 1
- all three `data/processed/sst_forecast_data*.csv` files from notebook 3

**Outputs**
- Episode-by-episode forecast tables and MAE / RMSE summaries (printed)

Run this notebook from the `notebooks/` folder, top to bottom. It takes about 40 seconds, most of it BSTS sampling.

## Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Coastal El Niño / La Niña rule (same as notebook 1):
# El Niño when ICEN >= +0.4 °C, La Niña when ICEN <= -0.4 °C, for at least 3 consecutive months.
ENSO_WARM, ENSO_COLD, ENSO_MIN_MONTHS = 0.4, -0.4, 3

# Relabelled ICEN written by 1_enso_episodes.ipynb
ICEN_CSV = Path("../data/processed/icen_monthly_pm04.csv")
assert ICEN_CSV.exists(), "Run 1_enso_episodes.ipynb first"

## 4. Baseline models: remaining episode duration (`forecast_episode_duration.py`)

Compares the historical median, linear regression, ARIMA, ETS and BSTS. Uses ICEN only. Episodes follow the ±0.4 °C, 3-month rule from the setup cell.

In [2]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.arima.model import ARIMA

ICEN_DELAY = 2
MIN_TRAIN_EPISODES = 3
MIN_ICEN_MONTHS = 24
ARIMA_ORDER = (1, 0, 1)
MAX_ARIMA_STEPS = 36

STUDY_START = pd.Period("2001-01", freq="M")
STUDY_END = pd.Period("2022-12", freq="M")

FEATURES = ["ICEN at confirmation"]
TARGET = "Actual months left"
DURATION_METHODS = ["Median", "Regression", "ARIMA", "ETS", "BSTS"]

# ---- ETS and BSTS duration forecasts (added) -------------------------------------
# Both forecast the monthly ICEN path from the history available on the issue date.
# Neither pulls back to normal on its own (BSTS's level is a random walk), so instead of one
# average path they simulate N_PATHS possible futures. The episode is predicted to end at the
# median month in which the simulated ICEN first leaves the episode range (below ENSO_WARM for
# El Niño, above ENSO_COLD for La Niña). If the median path never leaves within the window,
# the forecast is capped at the window end and still scored.
import warnings
from statsmodels.tsa.exponential_smoothing.ets import ETSModel

N_PATHS = 1000
BSTS_DRAWS, BSTS_BURN = 3000, 1000


def in_phase(values, phase):
    return values >= ENSO_WARM if phase == "El Niño" else values <= ENSO_COLD


def months_left_from_paths(paths, phase, forecast_month):
    """paths: DataFrame indexed by future months, one column per simulated path."""
    outside = ~in_phase(paths.to_numpy(), phase)
    first_exit = np.where(outside.any(axis=0), outside.argmax(axis=0), np.inf)
    median_exit = np.quantile(first_exit, 0.5, method="lower")
    if np.isinf(median_exit):
        # The median path never leaves the episode range: the model says the episode lasts at
        # least to the end of the window, so the forecast is capped there and still scored.
        return max(0, paths.index[-1].ordinal - forecast_month.ordinal), "Scored (capped at window end)"
    last_episode_month = paths.index[int(median_exit)] - 1
    return max(0, last_episode_month.ordinal - forecast_month.ordinal), "Scored"


def ets_paths(history, steps, seed):
    """ETS with additive errors; no trend, additive or damped trend chosen by AIC; simulated paths."""
    fits = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for trend, damped in [(None, False), ("add", False), ("add", True)]:
            try:
                fits.append(ETSModel(history, error="add", trend=trend, damped_trend=damped).fit(disp=False))
            except (ValueError, np.linalg.LinAlgError):
                continue
        best = min(fits, key=lambda f: f.aic)
        sims = best.simulate(nsimulations=steps, repetitions=N_PATHS, anchor="end",
                              rng=np.random.default_rng(seed))
    future = pd.period_range(history.index[-1] + 1, periods=steps, freq="M")
    return pd.DataFrame(np.asarray(sims), index=future)


def bsts_paths(history, steps, seed, draws=BSTS_DRAWS, burn=BSTS_BURN):
    """Local-level BSTS fitted by Gibbs sampling (bsts-style default priors); posterior predictive paths.

    y_t = mu_t + e_t, e_t ~ N(0, s_e^2);  mu_t = mu_{t-1} + u_t, u_t ~ N(0, s_u^2)
    """
    rng = np.random.default_rng(seed)
    y = history.to_numpy(float); n = len(y); sdy = y.std(ddof=1)
    a0, se_b, su_b = 0.005, 0.005 * sdy ** 2, 0.005 * (0.01 * sdy) ** 2
    se2, su2 = sdy ** 2 / 4, (0.1 * sdy) ** 2
    kept = draws - burn
    keep_every = max(1, kept // N_PATHS)
    paths = []
    m, P = np.empty(n), np.empty(n)
    for it in range(draws):
        # Forward filter
        m_prev, P_prev = y[0], sdy ** 2
        for t in range(n):
            Pp = P_prev + (su2 if t else 0.0)
            K = Pp / (Pp + se2)
            m_prev = m_prev + K * (y[t] - m_prev)
            P_prev = (1 - K) * Pp
            m[t], P[t] = m_prev, P_prev
        # Backward sample the level
        z = rng.standard_normal(n)
        mu = np.empty(n)
        mu[-1] = m[-1] + np.sqrt(P[-1]) * z[-1]
        for t in range(n - 2, -1, -1):
            J = P[t] / (P[t] + su2)
            mu[t] = m[t] + J * (mu[t + 1] - m[t]) + np.sqrt((1 - J) * P[t]) * z[t]
        resid = y - mu
        se2 = 1 / rng.gamma(a0 + n / 2, 1 / (se_b + resid @ resid / 2))
        d = np.diff(mu)
        su2 = min(1 / rng.gamma(a0 + (n - 1) / 2, 1 / (su_b + d @ d / 2)), sdy ** 2)
        if it >= burn and (it - burn) % keep_every == 0:
            level = mu[-1] + np.cumsum(rng.normal(0, np.sqrt(su2), steps))
            paths.append(level + rng.normal(0, np.sqrt(se2), steps))
    future = pd.period_range(history.index[-1] + 1, periods=steps, freq="M")
    return pd.DataFrame(np.column_stack(paths), index=future)


def load_icen():
    icen = pd.read_csv(ICEN_CSV)
    dates = pd.to_datetime(
        icen[["Year", "Month"]]
        .rename(columns={"Year": "year", "Month": "month"})
        .assign(day=1)
    )
    icen["month"] = dates.dt.to_period("M")
    return icen.sort_values("month")


def build_episode_table(icen):
    episodes = []

    for episode_id, event in icen.dropna(subset=["Episode_ID"]).groupby("Episode_ID"):
        start = event["month"].iloc[0]
        end = event["month"].iloc[-1]

        if start < STUDY_START or end > STUDY_END:
            continue

        # The third qualifying ICEN value confirms the episode.
        confirmation = event.iloc[2]
        forecast_month = confirmation["month"] + ICEN_DELAY

        episodes.append({
            "Episode_ID": int(episode_id),
            "Phase": confirmation["Episode"],
            "Start": start,
            "End": end,
            "Forecast month": forecast_month,
            "End known month": end + 1 + ICEN_DELAY,
            "ICEN at confirmation": float(confirmation["ICEN"]),
            TARGET: max(0, end.ordinal - forecast_month.ordinal),
        })

    return pd.DataFrame(episodes).sort_values("Forecast month").reset_index(drop=True)


def make_forecasts(results, icen):
    series = icen.set_index("month")["ICEN"]
    calendar = pd.period_range(series.index.min(), series.index.max(), freq="M")
    series = series.reindex(calendar)

    results["Past episodes used"] = 0
    results["Median months left"] = float("nan")
    results["Regression months left"] = float("nan")
    results["ARIMA months left"] = float("nan")
    results["ARIMA status"] = "Insufficient ICEN history"
    for method in ["ETS", "BSTS"]:
        results[f"{method} months left"] = float("nan")
        results[f"{method} status"] = "Insufficient ICEN history"

    for i, episode in results.iterrows():
        forecast_month = episode["Forecast month"]
        phase = episode["Phase"]

        # The benchmarks learn from same-phase episodes with known end dates.
        past = results.loc[
            results["Phase"].eq(phase)
            & results["End known month"].le(forecast_month)
        ]
        results.loc[i, "Past episodes used"] = len(past)

        if not past.empty:
            results.loc[i, "Median months left"] = past[TARGET].median()

        if len(past) >= MIN_TRAIN_EPISODES:
            model = LinearRegression().fit(past[FEATURES], past[TARGET])
            prediction = model.predict(results.loc[[i], FEATURES])[0]
            results.loc[i, "Regression months left"] = max(0.0, prediction)

        # At the issue date, only ICEN through the confirmation month is available.
        last_available_month = forecast_month - ICEN_DELAY
        history = series.loc[:last_available_month]

        if history.count() < MIN_ICEN_MONTHS:
            continue

        # ETS and BSTS: median exit month over simulated ICEN paths
        for method, simulate in [("ETS", ets_paths), ("BSTS", bsts_paths)]:
            paths = simulate(history, MAX_ARIMA_STEPS, seed=int(episode["Episode_ID"]))
            left, status = months_left_from_paths(paths, phase, forecast_month)
            results.loc[i, f"{method} months left"] = left
            results.loc[i, f"{method} status"] = status

        fitted = ARIMA(history, order=ARIMA_ORDER).fit()
        if not fitted.mle_retvals.get("converged", True):
            results.loc[i, "ARIMA status"] = "Fit did not converge"
            continue

        path = fitted.forecast(steps=MAX_ARIMA_STEPS)
        outside_phase = ~in_phase(path, phase)   # ±0.4 °C rule (was 0.5 / -0.7)
        exit_months = path.index[outside_phase]

        if exit_months.empty:
            results.loc[i, "ARIMA status"] = "No exit within forecast window"
            continue

        # Count only episode months after the forecast month.
        last_episode_month = exit_months[0] - 1
        results.loc[i, "ARIMA months left"] = max(
            0, last_episode_month.ordinal - forecast_month.ordinal
        )
        results.loc[i, "ARIMA status"] = "Scored"

    for method in DURATION_METHODS:
        results[f"{method} error"] = (
            results[f"{method} months left"] - results[TARGET]
        ).abs()

    return results


def main():
    icen = load_icen()
    results = make_forecasts(build_episode_table(icen), icen)

    print("\nDuration forecasts by episode")
    print(
        results.drop(columns="End known month").to_string(
            index=False, float_format="{:.2f}".format, na_rep="—"
        )
    )

    print("\nForecast status by method")
    status_cols = {m: results[f"{m} status"].value_counts() for m in ["ARIMA", "ETS", "BSTS"]}
    print(pd.DataFrame(status_cols).fillna(0).astype(int).to_string())

    # Score all methods on exactly the same episodes.
    shared = results.dropna(subset=[f"{m} error" for m in DURATION_METHODS])

    print("\nDuration MAE and RMSE in months on the same episodes")
    if shared.empty:
        print("No episodes had forecasts from all methods.")
        return

    scores = {"Episodes": shared.groupby("Phase")["Episode_ID"].size()}
    for m in DURATION_METHODS:
        scores[f"{m}_MAE"] = shared.groupby("Phase")[f"{m} error"].mean()
        scores[f"{m}_RMSE"] = shared.groupby("Phase")[f"{m} error"].agg(lambda e: e.pow(2).mean() ** 0.5)
    summary = pd.DataFrame(scores)
    summary.loc["All"] = [len(shared)] + [
        shared[f"{m} error"].mean() if k == "MAE" else shared[f"{m} error"].pow(2).mean() ** 0.5
        for m in DURATION_METHODS for k in ["MAE", "RMSE"]]
    print(summary.to_string(float_format="{:.2f}".format))

In [3]:
main()

.venv/lib/python3.11/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
.venv/lib/python3.11/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params



Duration forecasts by episode
 Episode_ID   Phase   Start     End Forecast month  ICEN at confirmation  Actual months left  Past episodes used  Median months left  Regression months left  ARIMA months left                   ARIMA status  ETS months left                    ETS status  BSTS months left                   BSTS status  Median error  Regression error  ARIMA error  ETS error  BSTS error
          1 La Niña 2001-06 2002-01        2001-10                 -0.88                   3                   0                   —                       —                  —      Insufficient ICEN history                —     Insufficient ICEN history                 —     Insufficient ICEN history             —                 —            —          —           —
          2 El Niño 2002-03 2002-05        2002-07                  0.67                   0                   0                   —                       —                  —      Insufficient ICEN history                —     I

## 5. Peak SST over the rest of an episode: ARIMA + GAM vs. persistence and historical median (`forecast_peak_sst_arima_gam.py`)

Uses `build_episode_table`, `load_icen`, `make_forecasts` and `ICEN_DELAY` from Section 4. Reads all three `sst_forecast_data*.csv` files from Section 2.

`FEATURES` is renamed `PEAK_FEATURES` here, because Section 4's `make_forecasts` reads the global `FEATURES`. This cell redefines `main`, so re-run Section 4's code cell before calling its `main()` again.

In [4]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer

FIRST_FORECAST = pd.Period("2011-01", freq="M")
MIDDLE_PERIOD_START = pd.Period("2014-01", freq="M")
LATER_PERIOD_START = pd.Period("2018-01", freq="M")

MIN_TRAIN_ROWS = 36
LINEAR_FEATURES = ["latest_sst", "previous_sst", "month_sin", "month_cos"]
PEAK_FEATURES = [*LINEAR_FEATURES, "available_icen"]


def load_sst(filename):
    data = pd.read_csv(filename)
    dates = pd.to_datetime(data[["year", "month"]].assign(day=1))
    data.index = pd.PeriodIndex(dates, freq="M")

    sst = data["sst_anom"].where(data["low_coverage"].eq(False)).sort_index()
    calendar = pd.period_range(sst.index.min(), sst.index.max(), freq="M")
    return sst.reindex(calendar)


def make_gam():
    terms = ColumnTransformer([
        ("linear", "passthrough", LINEAR_FEATURES),
        (
            "icen_curve",
            SplineTransformer(
                n_knots=3,
                degree=2,
                include_bias=False,
                extrapolation="constant",
            ),
            ["available_icen"],
        ),
    ])
    return make_pipeline(terms, LinearRegression())


def forecast_sst_path(sst, icen, issue_month, months_left):
    """Fit a separate GAM for each lead time from the same issue month."""
    data = pd.DataFrame({"actual": sst, "ICEN": icen.reindex(sst.index)})
    angle = 2 * np.pi * (data.index.month - 1) / 12
    data["month_sin"] = np.sin(angle)
    data["month_cos"] = np.cos(angle)

    predictions = []

    for lead in range(1, months_left + 1):
        target_month = issue_month + lead
        if target_month not in data.index:
            return None, "Forecast extends beyond SST calendar"

        pairs = data.assign(
            latest_sst=data["actual"].shift(lead),
            previous_sst=data["actual"].shift(lead + 1),
            available_icen=data["ICEN"].shift(lead + ICEN_DELAY),
        )

        # Train only on SST outcomes available by the forecast month.
        train = pairs.loc[:issue_month].dropna(subset=["actual", *PEAK_FEATURES])
        inputs = pairs.loc[[target_month], PEAK_FEATURES]

        if len(train) < MIN_TRAIN_ROWS:
            return None, f"Too few training months at lead {lead}"
        if inputs.isna().any().any():
            return None, "Missing SST or ICEN at forecast date"

        gam = make_gam().fit(train[PEAK_FEATURES], train["actual"])
        predictions.append(gam.predict(inputs)[0])

    return np.asarray(predictions), "Forecast issued"


def observed_extreme(sst, issue_month, end_month, extreme):
    """Calculate the observed extreme over the remaining episode months."""
    if end_month <= issue_month:
        return np.nan

    months = pd.period_range(issue_month + 1, end_month, freq="M")
    observed = sst.reindex(months)

    if observed.isna().any():
        return np.nan

    return extreme(observed.to_numpy())


def historical_median(episodes, sst, phase, issue_month, extreme):
    # Only use earlier same-phase episodes whose end was already known.
    past = episodes.loc[
        episodes["Phase"].eq(phase)
        & episodes["End known month"].le(issue_month)
        & episodes["Actual months left"].gt(0)
    ]

    extremes = [
        observed_extreme(sst, event["Forecast month"], event["End"], extreme)
        for _, event in past.iterrows()
    ]
    usable = pd.Series(extremes, dtype=float).dropna()
    return usable.median(), len(usable)


def print_scores(results, forecast_columns, title):
    common = results.dropna(
        subset=["Actual extreme", *forecast_columns.values()]
    )

    print(f"\n{title} (°C)")
    if common.empty:
        print("No episodes had a usable observed extreme and all forecasts.")
        return

    scores = []

    for phase, cases in common.groupby("Phase"):
        for model, column in forecast_columns.items():
            errors = cases[column] - cases["Actual extreme"]
            scores.append({
                "Phase": phase,
                "Model": model,
                "Episodes": len(cases),
                "MAE": errors.abs().mean(),
                "RMSE": np.sqrt(errors.pow(2).mean()),
            })

    print(pd.DataFrame(scores).to_string(
        index=False, float_format="{:.3f}".format
    ))


def main():
    icen = load_icen()
    episodes = make_forecasts(build_episode_table(icen), icen)
    icen_series = icen.set_index("month")["ICEN"]

    sst_2011_2013 = load_sst("../data/processed/sst_forecast_data_2011_2013.csv")
    sst_2014_2017 = load_sst("../data/processed/sst_forecast_data_2014_2017.csv")
    sst_2018_2022 = load_sst("../data/processed/sst_forecast_data.csv")

    rows = []

    for _, episode in episodes.iterrows():
        issue = episode["Forecast month"]
        if issue < FIRST_FORECAST:
            continue

        # Choose an SST reference fitted before the forecast period began.
        if issue < MIDDLE_PERIOD_START:
            sst = sst_2011_2013
        elif issue < LATER_PERIOD_START:
            sst = sst_2014_2017
        else:
            sst = sst_2018_2022

        phase = episode["Phase"]
        extreme = np.max if phase == "El Niño" else np.min
        predicted_left = episode["ARIMA months left"]

        median, history_count = historical_median(
            episodes, sst, phase, issue, extreme
        )

        row = {
            "Episode_ID": episode["Episode_ID"],
            "Phase": phase,
            "Forecast month": issue,
            "Actual months left": int(episode["Actual months left"]),
            "ARIMA months left": predicted_left,
            "Actual extreme": np.nan,
            "GAM extreme": np.nan,
            "Persistence extreme": sst.get(issue, np.nan),
            "Historical median extreme": median,
            "Historical episodes used": history_count,
            "Status": "No ARIMA duration forecast",
        }

        if pd.notna(predicted_left):
            if predicted_left == 0:
                row["Status"] = "Predicted episode already over"
            else:
                path, status = forecast_sst_path(
                    sst, icen_series, issue, int(predicted_left)
                )
                row["Status"] = status
                if path is not None:
                    row["GAM extreme"] = extreme(path)

        # Use the actual end date only to calculate the observed outcome.
        row["Actual extreme"] = observed_extreme(
            sst, issue, episode["End"], extreme
        )
        rows.append(row)

    results = pd.DataFrame(rows)
    if results.empty:
        print("No episodes in the SST forecast period.")
        return

    print("\nRemaining SST extremes by episode (°C)")
    print(results.to_string(
        index=False, float_format="{:.3f}".format, na_rep="—"
    ))

    continued = results["Actual months left"].gt(0)
    gam_issued = results["GAM extreme"].notna()
    predicted_zero = results["ARIMA months left"].eq(0)

    print("\nForecast coverage")
    print(f"Episodes that actually continued: {continued.sum()}")
    print(
        "Predicted zero months left but continued: "
        f"{(predicted_zero & continued).sum()}"
    )
    print(
        "Any missing GAM forecast despite continuation: "
        f"{(~gam_issued & continued).sum()}"
    )
    print(
        "GAM forecast issued but no months actually remained: "
        f"{(gam_issued & ~continued).sum()}"
    )
    print(
        "Continuing episodes with incomplete observed SST: "
        f"{(continued & results['Actual extreme'].isna()).sum()}"
    )

    original_models = {
        "ARIMA + GAM": "GAM extreme",
        "Persistence": "Persistence extreme",
    }
    print_scores(results, original_models, "ARIMA + GAM vs. persistence")

    print_scores(
        results,
        {**original_models, "Historical median": "Historical median extreme"},
        "Three-way comparison on shared episodes",
    )

In [5]:
main()

.venv/lib/python3.11/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
.venv/lib/python3.11/site-packages/statsmodels/tsa/statespace/mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params



Remaining SST extremes by episode (°C)
 Episode_ID   Phase Forecast month  Actual months left  ARIMA months left  Actual extreme  GAM extreme  Persistence extreme  Historical median extreme  Historical episodes used          Status
         13 La Niña        2012-01                   0              5.000               —        0.122                0.309                     -0.480                         4 Forecast issued
         14 El Niño        2012-07                   0              3.000               —        0.487                0.629                      0.112                         2 Forecast issued
         15 La Niña        2013-03                   8              7.000          -0.364       -0.206               -0.192                     -0.480                         4 Forecast issued
         16 El Niño        2014-09                   2              3.000           0.503        0.180                0.051                      0.109                         2 Forecast is